In [1]:
# ----------------------------
# Cell 1 — Imports and paths
# ----------------------------
import gc
import numpy as np
import pandas as pd
import scanpy as sc
from pathlib import Path
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
RESULTS_DIR = PROJECT_DIR / "results" / "phase4_leaveoneout"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Setup complete")

Setup complete


In [2]:
# ----------------------------
# Cell 2 — Load raw counts + annotation for GSE114725 (using the
# already-corrected, MT/VDJ-clean checkpoint from earlier this week)
# ----------------------------
def load_raw_with_annotation(raw_path, annotated_path, dataset_name):
    raw = sc.read_h5ad(raw_path)
    annotated = sc.read_h5ad(annotated_path, backed="r")
    qc_passed_barcodes = annotated.obs_names
    raw_qc = raw[raw.obs_names.isin(qc_passed_barcodes)].copy()
    meta_cols = [c for c in annotated.obs.columns]
    raw_qc.obs = raw_qc.obs.join(annotated.obs[meta_cols], rsuffix="_annotated")
    del raw
    gc.collect()
    return raw_qc

adata1_raw = load_raw_with_annotation(
    PROCESSED_DIR / "GSE114725_phase1_v2_clean_rawcounts.h5ad",
    PROCESSED_DIR / "GSE114725_phase2_v2_annotated.h5ad",
    "GSE114725"
)
print(f"Loaded: {adata1_raw.n_obs} cells")
print(adata1_raw.obs["patient"].unique())

Loaded: 44662 cells
['BC5', 'BC7', 'BC6', 'BC2', 'BC4', 'BC8', 'BC3', 'BC1']
Categories (8, object): ['BC1', 'BC2', 'BC3', 'BC4', 'BC5', 'BC6', 'BC7', 'BC8']


In [3]:
# ----------------------------
# Cell 3 — Leave-one-out robustness check: Macrophages, Tumour vs
# Normal. Re-run the pseudobulk DE 8 times, each time excluding one
# patient, checking whether FN1/HSPA1A/HSPA1B remain significant and
# whether the fold-change stays stable.
# ----------------------------
def build_pseudobulk(adata_raw, sample_col, cell_type, min_cells=10):
    subset = adata_raw[adata_raw.obs["cell_type"] == cell_type]
    samples, ids = [], []
    for sid in subset.obs[sample_col].unique():
        mask = (subset.obs[sample_col] == sid).values
        if mask.sum() < min_cells:
            continue
        X = subset.X[mask]
        samples.append(np.asarray(X.sum(axis=0)).flatten())
        ids.append(sid)
    counts_df = pd.DataFrame(samples, index=ids, columns=subset.var_names).T
    meta_df = pd.DataFrame({sample_col: ids}, index=ids)
    return counts_df, meta_df

patients = adata1_raw.obs["patient"].cat.categories.tolist()
target_genes = ["FN1", "HSPA1A", "HSPA1B"]
loo_results = []

for excluded_patient in patients:
    adata_loo = adata1_raw[adata1_raw.obs["patient"] != excluded_patient].copy()

    counts_df, meta_df = build_pseudobulk(adata_loo, "patient", "Macrophages")
    tissue_lookup = adata_loo.obs.drop_duplicates("patient").set_index("patient")["tissue"]
    meta_df["tissue"] = meta_df["patient"].map(tissue_lookup)
    meta_sub = meta_df[meta_df["tissue"].isin(["TUMOR", "NORMAL"])].copy()
    meta_sub["tissue"] = meta_sub["tissue"].astype(str)
    counts_sub = counts_df[meta_sub.index]

    n_tumor = (meta_sub["tissue"] == "TUMOR").sum()
    n_normal = (meta_sub["tissue"] == "NORMAL").sum()
    print(f"Excluding {excluded_patient}: {n_tumor} TUMOR / {n_normal} NORMAL samples remaining")

    if n_tumor < 2 or n_normal < 2:
        print(f"  SKIPPED — insufficient samples after exclusion")
        continue

    gene_filter = (counts_sub > 0).sum(axis=1) >= 10
    counts_sub = counts_sub[gene_filter]

    try:
        dds = DeseqDataSet(counts=counts_sub.T.astype(int), metadata=meta_sub,
                            design_factors="tissue", refit_cooks=True, quiet=True)
        dds.deseq2()
        ds = DeseqStats(dds, contrast=["tissue", "TUMOR", "NORMAL"], quiet=True)
        ds.summary()
        results = ds.results_df.copy()

        row = {"excluded_patient": excluded_patient, "n_tumor": n_tumor, "n_normal": n_normal}
        for gene in target_genes:
            if gene in results.index:
                row[f"{gene}_log2FC"] = results.loc[gene, "log2FoldChange"]
                row[f"{gene}_padj"] = results.loc[gene, "padj"]
                row[f"{gene}_significant"] = results.loc[gene, "padj"] < 0.05
            else:
                row[f"{gene}_log2FC"] = None
                row[f"{gene}_significant"] = False
        loo_results.append(row)
    except Exception as e:
        print(f"  FAILED — {e}")

    del adata_loo
    gc.collect()

loo_df = pd.DataFrame(loo_results)
print("\n=== Leave-one-out results ===")
print(loo_df.to_string(index=False))
loo_df.to_csv(RESULTS_DIR / "GSE114725_macrophage_leaveoneout.csv", index=False)

Excluding BC1: 3 TUMOR / 2 NORMAL samples remaining
  FAILED — no types given
Excluding BC2: 3 TUMOR / 2 NORMAL samples remaining
  FAILED — no types given
Excluding BC3: 3 TUMOR / 1 NORMAL samples remaining
  SKIPPED — insufficient samples after exclusion
Excluding BC4: 3 TUMOR / 2 NORMAL samples remaining
  FAILED — no types given
Excluding BC5: 2 TUMOR / 2 NORMAL samples remaining
  FAILED — no types given
Excluding BC6: 2 TUMOR / 2 NORMAL samples remaining
  FAILED — no types given
Excluding BC7: 3 TUMOR / 1 NORMAL samples remaining
  SKIPPED — insufficient samples after exclusion
Excluding BC8: 2 TUMOR / 2 NORMAL samples remaining
  FAILED — no types given

=== Leave-one-out results ===
Empty DataFrame
Columns: []
Index: []


In [4]:
# Diagnostic — check what's actually going into DESeq2 for one iteration
excluded_patient = "BC1"
adata_loo = adata1_raw[adata1_raw.obs["patient"] != excluded_patient].copy()

counts_df, meta_df = build_pseudobulk(adata_loo, "patient", "Macrophages")
tissue_lookup = adata_loo.obs.drop_duplicates("patient").set_index("patient")["tissue"]
meta_df["tissue"] = meta_df["patient"].map(tissue_lookup)
meta_sub = meta_df[meta_df["tissue"].isin(["TUMOR", "NORMAL"])].copy()
meta_sub["tissue"] = meta_sub["tissue"].astype(str)
counts_sub = counts_df[meta_sub.index]

print("meta_sub:")
print(meta_sub)
print("\nmeta_sub tissue dtype:", meta_sub["tissue"].dtype)
print("meta_sub tissue unique values:", meta_sub["tissue"].unique())
print("\ncounts_sub shape:", counts_sub.shape)
print("counts_sub columns match meta_sub index:", list(counts_sub.columns) == list(meta_sub.index))

meta_sub:
    patient  tissue
BC5     BC5   TUMOR
BC7     BC7  NORMAL
BC6     BC6   TUMOR
BC8     BC8   TUMOR
BC3     BC3  NORMAL

meta_sub tissue dtype: object
meta_sub tissue unique values: ['TUMOR' 'NORMAL']

counts_sub shape: (14800, 5)
counts_sub columns match meta_sub index: True


In [5]:
# Run the actual DESeq2 call directly, without the try/except hiding
# the real error
gene_filter = (counts_sub > 0).sum(axis=1) >= 10
counts_sub_filtered = counts_sub[gene_filter]
print(f"Genes after filter: {counts_sub_filtered.shape}")

dds = DeseqDataSet(counts=counts_sub_filtered.T.astype(int), metadata=meta_sub,
                    design_factors="tissue", refit_cooks=True, quiet=True)
dds.deseq2()

Genes after filter: (0, 5)


ValueError: no types given

In [6]:
# ----------------------------
# FIX: gene filter threshold was fixed at 10 samples, but leave-one-out
# iterations only have ~5 samples total — no gene could pass. Changed
# to require expression in at least 3 samples (matching the min-group-
# size convention used throughout this project), not a fixed count of 10.
# ----------------------------
gene_filter = (counts_sub > 0).sum(axis=1) >= 3
counts_sub_filtered = counts_sub[gene_filter]
print(f"Genes after filter: {counts_sub_filtered.shape}")

dds = DeseqDataSet(counts=counts_sub_filtered.T.astype(int), metadata=meta_sub,
                    design_factors="tissue", refit_cooks=True, quiet=True)
dds.deseq2()
print("Success")

Genes after filter: (13333, 5)


C:\Users\annam\AppData\Local\Temp\ipykernel_32256\2789504403.py:11: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(counts=counts_sub_filtered.T.astype(int), metadata=meta_sub,
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\pydeseq2\dds.py:541: UserWarning: As the residual degrees of freedom is less than 3, the distribution of log dispersions is especially asymmetric and likely to be poorly estimated by the MAD.
  self.fit_dispersion_prior()


Success


In [7]:
# ----------------------------
# Cell 3 (corrected) — Leave-one-out robustness check: Macrophages,
# Tumour vs Normal. FIX: gene filter threshold changed from >=10
# samples to >=3 samples, since each leave-one-out iteration only has
# ~4-5 total samples (the original >=10 threshold was copied from the
# full-dataset DE, where it made sense with many more samples).
# ----------------------------
patients = adata1_raw.obs["patient"].cat.categories.tolist()
target_genes = ["FN1", "HSPA1A", "HSPA1B"]
loo_results = []

for excluded_patient in patients:
    adata_loo = adata1_raw[adata1_raw.obs["patient"] != excluded_patient].copy()

    counts_df, meta_df = build_pseudobulk(adata_loo, "patient", "Macrophages")
    tissue_lookup = adata_loo.obs.drop_duplicates("patient").set_index("patient")["tissue"]
    meta_df["tissue"] = meta_df["patient"].map(tissue_lookup)
    meta_sub = meta_df[meta_df["tissue"].isin(["TUMOR", "NORMAL"])].copy()
    meta_sub["tissue"] = meta_sub["tissue"].astype(str)
    counts_sub = counts_df[meta_sub.index]

    n_tumor = (meta_sub["tissue"] == "TUMOR").sum()
    n_normal = (meta_sub["tissue"] == "NORMAL").sum()
    print(f"Excluding {excluded_patient}: {n_tumor} TUMOR / {n_normal} NORMAL samples remaining")

    if n_tumor < 2 or n_normal < 2:
        print(f"  SKIPPED — insufficient samples after exclusion")
        continue

    gene_filter = (counts_sub > 0).sum(axis=1) >= 3
    counts_sub_filtered = counts_sub[gene_filter]

    try:
        dds = DeseqDataSet(counts=counts_sub_filtered.T.astype(int), metadata=meta_sub,
                            design_factors="tissue", refit_cooks=True, quiet=True)
        dds.deseq2()
        ds = DeseqStats(dds, contrast=["tissue", "TUMOR", "NORMAL"], quiet=True)
        ds.summary()
        results = ds.results_df.copy()

        row = {"excluded_patient": excluded_patient, "n_tumor": n_tumor, "n_normal": n_normal}
        for gene in target_genes:
            if gene in results.index:
                row[f"{gene}_log2FC"] = results.loc[gene, "log2FoldChange"]
                row[f"{gene}_padj"] = results.loc[gene, "padj"]
                row[f"{gene}_significant"] = results.loc[gene, "padj"] < 0.05
            else:
                row[f"{gene}_log2FC"] = None
                row[f"{gene}_significant"] = False
        loo_results.append(row)
        print(f"  SUCCESS")
    except Exception as e:
        print(f"  FAILED — {type(e).__name__}: {e}")

    del adata_loo
    gc.collect()

loo_df = pd.DataFrame(loo_results)
print("\n=== Leave-one-out results ===")
print(loo_df.to_string(index=False))
loo_df.to_csv(RESULTS_DIR / "GSE114725_macrophage_leaveoneout.csv", index=False)

Excluding BC1: 3 TUMOR / 2 NORMAL samples remaining


C:\Users\annam\AppData\Local\Temp\ipykernel_32256\3436733034.py:34: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(counts=counts_sub_filtered.T.astype(int), metadata=meta_sub,
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\pydeseq2\dds.py:541: UserWarning: As the residual degrees of freedom is less than 3, the distribution of log dispersions is especially asymmetric and likely to be poorly estimated by the MAD.
  self.fit_dispersion_prior()


  SUCCESS
Excluding BC2: 3 TUMOR / 2 NORMAL samples remaining


C:\Users\annam\AppData\Local\Temp\ipykernel_32256\3436733034.py:34: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(counts=counts_sub_filtered.T.astype(int), metadata=meta_sub,
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\pydeseq2\dds.py:541: UserWarning: As the residual degrees of freedom is less than 3, the distribution of log dispersions is especially asymmetric and likely to be poorly estimated by the MAD.
  self.fit_dispersion_prior()


  SUCCESS
Excluding BC3: 3 TUMOR / 1 NORMAL samples remaining
  SKIPPED — insufficient samples after exclusion
Excluding BC4: 3 TUMOR / 2 NORMAL samples remaining


...

  SUCCESS
Excluding BC6: 2 TUMOR / 2 NORMAL samples remaining


C:\Users\annam\AppData\Local\Temp\ipykernel_32256\3436733034.py:34: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(counts=counts_sub_filtered.T.astype(int), metadata=meta_sub,
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\pydeseq2\dds.py:541: UserWarning: As the residual degrees of freedom is less than 3, the distribution of log dispersions is especially asymmetric and likely to be poorly estimated by the MAD.
  self.fit_dispersion_prior()


  SUCCESS
Excluding BC7: 3 TUMOR / 1 NORMAL samples remaining
  SKIPPED — insufficient samples after exclusion
Excluding BC8: 2 TUMOR / 2 NORMAL samples remaining


C:\Users\annam\AppData\Local\Temp\ipykernel_32256\3436733034.py:34: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(counts=counts_sub_filtered.T.astype(int), metadata=meta_sub,
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\pydeseq2\dds.py:541: UserWarning: As the residual degrees of freedom is less than 3, the distribution of log dispersions is especially asymmetric and likely to be poorly estimated by the MAD.
  self.fit_dispersion_prior()


  SUCCESS

=== Leave-one-out results ===
excluded_patient  n_tumor  n_normal  FN1_log2FC  FN1_padj  FN1_significant  HSPA1A_log2FC  HSPA1A_padj  HSPA1A_significant  HSPA1B_log2FC  HSPA1B_padj  HSPA1B_significant
             BC1        3         2    3.075921  0.077732            False       3.652900     0.649771               False       3.364106     0.666903               False
             BC2        3         2    3.075921  0.077732            False       3.652900     0.649771               False       3.364106     0.666903               False
             BC4        3         2    3.075921  0.077732            False       3.652900     0.649771               False       3.364106     0.666903               False
             BC5        2         2    3.603931  0.000101             True       0.894969     0.767364               False       0.884376     0.755442               False
             BC6        2         2    1.850034  0.999801            False       4.158853     0.730969  

In [8]:
# ----------------------------
# Cell 1 — Imports and paths
# ----------------------------
import gc
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from pathlib import Path
from sccoda.util import comp_ana as scc_ana

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
RESULTS_DIR = PROJECT_DIR / "results" / "phase4_leaveoneout"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Metadata only — composition counting doesn't need the expression
# matrix at all, avoiding GSE176078's memory issues entirely
adata2_meta = sc.read_h5ad(
    PROCESSED_DIR / "GSE176078_phase2_v2_annotated_corrected.h5ad", backed="r"
)
obs_df = adata2_meta.obs[["orig.ident", "subtype", "cell_type"]].copy()
del adata2_meta
gc.collect()

print(f"Loaded metadata: {len(obs_df)} cells")
her2_samples = obs_df[obs_df["subtype"] == "HER2+"]["orig.ident"].unique()
print(f"HER2+ samples: {her2_samples.tolist()}")




Loaded metadata: 91425 cells
HER2+ samples: ['CID3586', 'CID3921', 'CID45171', 'CID3838', 'CID4066']


In [9]:
# ----------------------------
# Cell 2 — Leave-one-out robustness check: Memory T cells, both
# credible comparisons (HER2+_vs_ER+, TNBC_vs_HER2+). Excludes one
# HER2+ sample at a time (5 iterations), checking whether the credible
# effect for Memory T cells survives.
# ----------------------------
def run_sccoda_comparison(obs_subset, group_a, group_b, reference="PVL"):
    comp_df = obs_subset.groupby(["orig.ident", "subtype", "cell_type"], observed=True) \
        .size().reset_index(name="count")
    comp_wide = comp_df.pivot_table(
        index=["orig.ident", "subtype"], columns="cell_type", values="count", fill_value=0
    ).reset_index()

    cell_type_cols = [c for c in comp_wide.columns if c not in ["orig.ident", "subtype"]]
    sccoda_data = ad.AnnData(
        X=comp_wide[cell_type_cols].values.astype(float),
        obs=comp_wide[["orig.ident", "subtype"]],
        var=pd.DataFrame(index=cell_type_cols)
    )
    sccoda_data.obs["subtype"] = sccoda_data.obs["subtype"].astype(str)

    mask = sccoda_data.obs["subtype"].isin([group_a, group_b])
    sccoda_sub = sccoda_data[mask].copy()
    sccoda_sub.obs["subtype"] = sccoda_sub.obs["subtype"].astype(str)

    model = scc_ana.CompositionalAnalysis(sccoda_sub, formula="subtype", reference_cell_type=reference)
    results = model.sample_hmc()
    credible = results.credible_effects()
    return credible

her2_samples_list = her2_samples.tolist()
loo_sccoda_results = []

for excluded_sample in her2_samples_list:
    obs_loo = obs_df[obs_df["orig.ident"] != excluded_sample].copy()

    n_her2_remaining = (obs_loo["subtype"] == "HER2+")["orig.ident" if False else True].sum() \
        if False else obs_loo[obs_loo["subtype"] == "HER2+"]["orig.ident"].nunique()
    print(f"\nExcluding {excluded_sample}: {n_her2_remaining} HER2+ samples remaining")

    row = {"excluded_sample": excluded_sample}
    for group_a, group_b in [("HER2+", "ER+"), ("TNBC", "HER2+")]:
        comp_name = f"{group_a}_vs_{group_b}"
        try:
            credible = run_sccoda_comparison(obs_loo, group_a, group_b)
            memory_t_credible = credible.loc["Memory T cells"] if "Memory T cells" in credible.index else None
            print(f"  {comp_name}: Memory T cells credible = {memory_t_credible}")
            row[f"{comp_name}_credible"] = bool(memory_t_credible) if memory_t_credible is not None else None
        except Exception as e:
            print(f"  {comp_name}: FAILED — {type(e).__name__}: {e}")
            row[f"{comp_name}_credible"] = "FAILED"

    loo_sccoda_results.append(row)
    del obs_loo
    gc.collect()

loo_sccoda_df = pd.DataFrame(loo_sccoda_results)
print("\n=== Leave-one-out results (HER2+ Memory T cells) ===")
print(loo_sccoda_df.to_string(index=False))
loo_sccoda_df.to_csv(RESULTS_DIR / "GSE176078_MemoryT_leaveoneout.csv", index=False)


Excluding CID3586: 4 HER2+ samples remaining


C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\anndata\_core\aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Zero counts encountered in data! Added a pseudocount of 0.5.


100%|██████████| 20000/20000 [02:32<00:00, 131.28it/s]


MCMC sampling finished. (194.868 sec)
Acceptance rate: 54.7%
  HER2+_vs_ER+: Memory T cells credible = None
Zero counts encountered in data! Added a pseudocount of 0.5.


...

MCMC sampling finished. (214.597 sec)
Acceptance rate: 54.9%
  TNBC_vs_HER2+: Memory T cells credible = None

Excluding CID4066: 4 HER2+ samples remaining
Zero counts encountered in data! Added a pseudocount of 0.5.


C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\anndata\_core\aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
100%|██████████| 20000/20000 [02:45<00:00, 120.70it/s]


MCMC sampling finished. (211.761 sec)
Acceptance rate: 49.9%
  HER2+_vs_ER+: Memory T cells credible = None
Zero counts encountered in data! Added a pseudocount of 0.5.


C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\anndata\_core\aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
100%|██████████| 20000/20000 [02:54<00:00, 114.59it/s]


MCMC sampling finished. (214.147 sec)
Acceptance rate: 54.8%
  TNBC_vs_HER2+: Memory T cells credible = None

=== Leave-one-out results (HER2+ Memory T cells) ===
excluded_sample HER2+_vs_ER+_credible TNBC_vs_HER2+_credible
        CID3586                  None                   None
        CID3921                  None                   None
       CID45171                  None                   None
        CID3838                  None                   None
        CID4066                  None                   None


In [10]:
# ----------------------------
# Cell 1 — Imports and paths
# ----------------------------
import gc
import numpy as np
import pandas as pd
import scanpy as sc
import gseapy as gp
from pathlib import Path
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
RESULTS_DIR = PROJECT_DIR / "results" / "phase4_leaveoneout"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [11]:
# ----------------------------
# Cell 2 — Load raw counts + annotation for GSE176078
# ----------------------------
def load_raw_with_annotation(raw_path, annotated_path, dataset_name):
    raw = sc.read_h5ad(raw_path)
    annotated = sc.read_h5ad(annotated_path, backed="r")
    qc_passed_barcodes = annotated.obs_names
    raw_qc = raw[raw.obs_names.isin(qc_passed_barcodes)].copy()
    meta_cols = [c for c in annotated.obs.columns]
    raw_qc.obs = raw_qc.obs.join(annotated.obs[meta_cols], rsuffix="_annotated")
    del raw
    gc.collect()
    return raw_qc

adata2_raw = load_raw_with_annotation(
    PROCESSED_DIR / "GSE176078_phase1_v2_clean_rawcounts.h5ad",
    PROCESSED_DIR / "GSE176078_phase2_v2_annotated_corrected.h5ad",
    "GSE176078"
)
print(f"Loaded: {adata2_raw.n_obs} cells")

tnbc_samples = adata2_raw.obs[adata2_raw.obs["subtype"] == "TNBC"]["orig.ident"].unique().tolist()
print(f"TNBC samples ({len(tnbc_samples)}): {tnbc_samples}")

Loaded: 91425 cells
TNBC samples (10): ['CID44041', 'CID4465', 'CID4495', 'CID44971', 'CID44991', 'CID4513', 'CID4515', 'CID4523', 'CID3946', 'CID3963']


In [15]:
# ----------------------------
# Cell 3 — Leave-one-out robustness check: B cells, TNF-alpha/NF-kB
# pathway, both credible comparisons. Excludes one TNBC sample at a
# time (10 iterations), checking whether the pathway remains
# significant in each.
# ----------------------------
def build_pseudobulk(adata_raw, sample_col, cell_type, min_cells=10):
    subset = adata_raw[adata_raw.obs["cell_type"] == cell_type]
    samples, ids = [], []
    for sid in subset.obs[sample_col].unique():
        mask = (subset.obs[sample_col] == sid).values
        if mask.sum() < min_cells:
            continue
        X = subset.X[mask]
        if hasattr(X, "toarray"): X = X.toarray()
        samples.append(np.asarray(X).sum(axis=0).flatten())
        ids.append(sid)
    counts_df = pd.DataFrame(samples, index=ids, columns=subset.var_names).T
    meta_df = pd.DataFrame({sample_col: ids}, index=ids)
    return counts_df, meta_df

def run_de_and_check_pathway(adata_raw, group_a, group_b, cell_type="B cells"):
    counts_df, meta_df = build_pseudobulk(adata_raw, "orig.ident", cell_type)
    subtype_lookup = adata_raw.obs.drop_duplicates("orig.ident").set_index("orig.ident")["subtype"]
    meta_df["subtype"] = meta_df["orig.ident"].map(subtype_lookup)
    meta_sub = meta_df[meta_df["subtype"].isin([group_a, group_b])].copy()
    meta_sub["subtype"] = meta_sub["subtype"].astype(str)
    counts_sub = counts_df[meta_sub.index]

    n_a = (meta_sub["subtype"] == group_a).sum()
    n_b = (meta_sub["subtype"] == group_b).sum()
    if n_a < 2 or n_b < 2:
        return None, f"insufficient samples ({n_a}/{n_b})"

    gene_filter = (counts_sub > 0).sum(axis=1) >= 3
    counts_sub_filtered = counts_sub[gene_filter]

    dds = DeseqDataSet(counts=counts_sub_filtered.T.astype(int), metadata=meta_sub,
                        design_factors="subtype", refit_cooks=True, quiet=True)
    dds.deseq2()
    ds = DeseqStats(dds, contrast=["subtype", group_a, group_b], quiet=True)
    ds.summary()
    de_results = ds.results_df.copy()
    sig_genes = de_results[de_results["padj"] < 0.05].index.tolist()

    if len(sig_genes) < 5:
        return None, f"only {len(sig_genes)} sig genes"

    enr = gp.enrich(gene_list=sig_genes, gene_sets="MSigDB_Hallmark_2020",
                    background=de_results.index.tolist(), outdir=None)
    tnf_row = enr.results[enr.results["Term"].str.contains("TNF-alpha", case=False)]
    tnf_significant = (tnf_row["Adjusted P-value"].astype(float) < 0.05).any() if len(tnf_row) > 0 else False
    tnf_pval = tnf_row["Adjusted P-value"].astype(float).min() if len(tnf_row) > 0 else None
    return {"n_sig_genes": len(sig_genes), "tnf_significant": tnf_significant, "tnf_pval": tnf_pval}, None


loo_bcell_results = []
for excluded_sample in tnbc_samples:
    adata_loo = adata2_raw[adata2_raw.obs["orig.ident"] != excluded_sample].copy()
    print(f"\nExcluding {excluded_sample}")


Excluding CID44041

Excluding CID4465

Excluding CID4495

Excluding CID44971

Excluding CID44991

Excluding CID4513

Excluding CID4515

Excluding CID4523

Excluding CID3946

Excluding CID3963


In [14]:
print("run_de_and_check_pathway" in dir())
print("build_pseudobulk" in dir())

True
True


In [16]:
adata_test = adata2_raw[adata2_raw.obs["orig.ident"] != "CID44041"].copy()
result, skip_reason = run_de_and_check_pathway(adata_test, "TNBC", "ER+")
print("Result:", result)
print("Skip reason:", skip_reason)

C:\Users\annam\AppData\Local\Temp\ipykernel_32256\4254860270.py:38: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(counts=counts_sub_filtered.T.astype(int), metadata=meta_sub,
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\pydeseq2\dds.py:807: UserWarning: The dispersion trend curve fitting did not converge. Switching to a mean-based dispersion trend.
  self._fit_parametric_dispersion_trend(vst)
Fitting dispersions...
... done in 0.21 seconds.

Fitting MAP dispersions...
... done in 0.24 seconds.

Fitting LFCs...
... done in 0.25 seconds.



Result: {'n_sig_genes': 39, 'tnf_significant': np.True_, 'tnf_pval': np.float64(6.639274070425579e-08)}
Skip reason: None


In [17]:
loo_bcell_results = []
for excluded_sample in tnbc_samples:
    adata_loo = adata2_raw[adata2_raw.obs["orig.ident"] != excluded_sample].copy()
    print(f"\nExcluding {excluded_sample}", flush=True)

    row = {"excluded_sample": excluded_sample}
    for group_a, group_b in [("TNBC", "ER+"), ("TNBC", "HER2+")]:
        comp_name = f"{group_a}_vs_{group_b}"
        print(f"  Starting {comp_name}...", flush=True)
        try:
            result, skip_reason = run_de_and_check_pathway(adata_loo, group_a, group_b)
            print(f"  Finished {comp_name}: result={result is not None}", flush=True)
            if result:
                row[f"{comp_name}_tnf_significant"] = result["tnf_significant"]
                row[f"{comp_name}_n_sig_genes"] = result["n_sig_genes"]
            else:
                row[f"{comp_name}_tnf_significant"] = None
        except Exception as e:
            print(f"  {comp_name}: FAILED — {type(e).__name__}: {e}", flush=True)
            row[f"{comp_name}_tnf_significant"] = "FAILED"

    loo_bcell_results.append(row)
    del adata_loo
    gc.collect()
    print(f"Completed {excluded_sample}, results so far: {len(loo_bcell_results)}", flush=True)

loo_bcell_df = pd.DataFrame(loo_bcell_results)
print("\n=== Leave-one-out results (B cell TNF-alpha) ===")
print(loo_bcell_df.to_string(index=False))
loo_bcell_df.to_csv(RESULTS_DIR / "GSE176078_Bcell_TNF_leaveoneout.csv", index=False)

MemoryError: Unable to allocate 649. MiB for an array with shape (170002971,) and data type float32

In [4]:
import gc
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from pathlib import Path
from sccoda.util import comp_ana as scc_ana

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
RESULTS_DIR = PROJECT_DIR / "results" / "phase4_leaveoneout"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ---- Load metadata ----
adata2_meta = sc.read_h5ad(
    PROCESSED_DIR / "GSE176078_phase2_v2_annotated_corrected.h5ad", backed="r"
)
obs_df = adata2_meta.obs[["orig.ident", "subtype", "cell_type"]].copy()
del adata2_meta
gc.collect()

her2_samples = obs_df[obs_df["subtype"] == "HER2+"]["orig.ident"].unique()
print(f"HER2+ samples: {her2_samples.tolist()}")

# ---- Helper functions ----
def run_sccoda_comparison(obs_subset, group_a, group_b, reference="PVL"):
    comp_df = obs_subset.groupby(["orig.ident", "subtype", "cell_type"], observed=True) \
        .size().reset_index(name="count")
    comp_wide = comp_df.pivot_table(
        index=["orig.ident", "subtype"], columns="cell_type", values="count", fill_value=0
    ).reset_index()
    cell_type_cols = [c for c in comp_wide.columns if c not in ["orig.ident", "subtype"]]
    sccoda_data = ad.AnnData(
        X=comp_wide[cell_type_cols].values.astype(float),
        obs=comp_wide[["orig.ident", "subtype"]],
        var=pd.DataFrame(index=cell_type_cols)
    )
    sccoda_data.obs["subtype"] = sccoda_data.obs["subtype"].astype(str)
    mask = sccoda_data.obs["subtype"].isin([group_a, group_b])
    sccoda_sub = sccoda_data[mask].copy()
    sccoda_sub.obs["subtype"] = sccoda_sub.obs["subtype"].astype(str)
    model = scc_ana.CompositionalAnalysis(sccoda_sub, formula="subtype", reference_cell_type=reference)
    results = model.sample_hmc()
    return results.credible_effects()

def get_credible(credible_series, cell_type):
    matches = [idx for idx in credible_series.index if idx[1] == cell_type]
    return bool(credible_series.loc[matches[0]]) if matches else None

# ---- Leave-one-out loop ----
her2_samples_list = her2_samples.tolist()
loo_sccoda_results = []

for excluded_sample in her2_samples_list:
    obs_loo = obs_df[obs_df["orig.ident"] != excluded_sample].copy()
    print(f"\nExcluding {excluded_sample}")
    row = {"excluded_sample": excluded_sample}
    for group_a, group_b in [("HER2+", "ER+"), ("TNBC", "HER2+")]:
        comp_name = f"{group_a}_vs_{group_b}"
        try:
            credible = run_sccoda_comparison(obs_loo, group_a, group_b)
            memory_t_credible = get_credible(credible, "Memory T cells")
            print(f"  {comp_name}: Memory T cells credible = {memory_t_credible}")
            row[f"{comp_name}_credible"] = memory_t_credible
        except Exception as e:
            print(f"  {comp_name}: FAILED — {type(e).__name__}: {e}")
            row[f"{comp_name}_credible"] = "FAILED"
    loo_sccoda_results.append(row)
    del obs_loo
    gc.collect()

loo_sccoda_df = pd.DataFrame(loo_sccoda_results)
print("\n=== Leave-one-out results (HER2+ Memory T cells) ===")
print(loo_sccoda_df.to_string(index=False))
loo_sccoda_df.to_csv(RESULTS_DIR / "GSE176078_MemoryT_leaveoneout.csv", index=False)

HER2+ samples: ['CID3586', 'CID3921', 'CID45171', 'CID3838', 'CID4066']

Excluding CID3586
Zero counts encountered in data! Added a pseudocount of 0.5.


C:\Users\annam\AppData\Local\Temp\ipykernel_22328\759860949.py:29: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  comp_wide = comp_df.pivot_table(
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\anndata\_core\aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
100%|██████████| 20000/20000 [02:30<00:00, 132.54it/s]


MCMC sampling finished. (194.197 sec)
Acceptance rate: 61.6%
  HER2+_vs_ER+: Memory T cells credible = False
Zero counts encountered in data! Added a pseudocount of 0.5.


C:\Users\annam\AppData\Local\Temp\ipykernel_22328\759860949.py:29: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  comp_wide = comp_df.pivot_table(
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\anndata\_core\aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
100%|██████████| 20000/20000 [02:21<00:00, 141.51it/s]


MCMC sampling finished. (180.893 sec)
Acceptance rate: 54.6%
  TNBC_vs_HER2+: Memory T cells credible = False

Excluding CID3921
Zero counts encountered in data! Added a pseudocount of 0.5.


...

MCMC sampling finished. (190.028 sec)
Acceptance rate: 58.8%
  TNBC_vs_HER2+: Memory T cells credible = False

Excluding CID4066
Zero counts encountered in data! Added a pseudocount of 0.5.


C:\Users\annam\AppData\Local\Temp\ipykernel_22328\759860949.py:29: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  comp_wide = comp_df.pivot_table(
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\anndata\_core\aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
100%|██████████| 20000/20000 [02:32<00:00, 130.90it/s]


MCMC sampling finished. (193.279 sec)
Acceptance rate: 56.3%
  HER2+_vs_ER+: Memory T cells credible = True
Zero counts encountered in data! Added a pseudocount of 0.5.


C:\Users\annam\AppData\Local\Temp\ipykernel_22328\759860949.py:29: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  comp_wide = comp_df.pivot_table(
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\anndata\_core\aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
100%|██████████| 20000/20000 [02:33<00:00, 130.14it/s]


MCMC sampling finished. (193.270 sec)
Acceptance rate: 61.1%
  TNBC_vs_HER2+: Memory T cells credible = True

=== Leave-one-out results (HER2+ Memory T cells) ===
excluded_sample  HER2+_vs_ER+_credible  TNBC_vs_HER2+_credible
        CID3586                  False                   False
        CID3921                  False                   False
       CID45171                  False                   False
        CID3838                  False                   False
        CID4066                   True                    True
